In [1]:
import pandas as pd
import numpy as np
from scipy import stats

pd.set_option("display.max_columns", None)

df = pd.read_csv("../data/cleaned/playstore_features.csv")
print(df.shape)   # expect (9658, 22)

(9658, 22)


In [2]:
ct = pd.crosstab(df["Category"], df["Success"])

chi2, p_value, dof, expected = stats.chi2_contingency(ct)

# Effect size for chi-square: Cramér's V
n = ct.sum().sum()
k = min(ct.shape) - 1
cramers_v = np.sqrt(chi2 / (n * k))

print(f"Chi-square statistic: {chi2:.2f}")
print(f"Degrees of freedom: {dof}")
print(f"p-value: {p_value:.2e}")
print(f"Cramér's V (effect size): {cramers_v:.3f}")

Chi-square statistic: 865.28
Degrees of freedom: 32
p-value: 3.53e-161
Cramér's V (effect size): 0.299


In [3]:
free_installs = df.loc[df["Type"] == "Free", "Installs"]
paid_installs = df.loc[df["Type"] == "Paid", "Installs"]

u_stat, p_value = stats.mannwhitneyu(free_installs, paid_installs, alternative="greater")

# Effect size: rank-biserial correlation
n1, n2 = len(free_installs), len(paid_installs)
rank_biserial = 1 - (2 * u_stat) / (n1 * n2)

print(f"Free apps median installs: {free_installs.median():,.0f}")
print(f"Paid apps median installs: {paid_installs.median():,.0f}")
print(f"U statistic: {u_stat:,.0f}")
print(f"p-value: {p_value:.2e}")
print(f"Rank-biserial correlation: {rank_biserial:.3f}")

Free apps median installs: 100,000
Paid apps median installs: 1,000
U statistic: 5,031,686
p-value: 3.34e-116
Rank-biserial correlation: -0.499


In [4]:
rng = np.random.default_rng(42)
a = free_installs.values
b = paid_installs.values

diffs = []
for _ in range(5000):
    sample_a = rng.choice(a, size=len(a), replace=True)
    sample_b = rng.choice(b, size=len(b), replace=True)
    diffs.append(np.median(sample_a) - np.median(sample_b))

ci_low, ci_high = np.percentile(diffs, [2.5, 97.5])
print(f"95% CI for (median free − median paid) installs: [{ci_low:,.0f}, {ci_high:,.0f}]")

95% CI for (median free − median paid) installs: [99,000, 99,000]


In [5]:
ct_update = pd.crosstab(df["Update_Bucket"], df["Success"])

chi2, p_value, dof, expected = stats.chi2_contingency(ct_update)
n = ct_update.sum().sum()
k = min(ct_update.shape) - 1
cramers_v = np.sqrt(chi2 / (n * k))

print(f"Chi-square: {chi2:.2f}, p-value: {p_value:.2e}, Cramér's V: {cramers_v:.3f}")

Chi-square: 1063.22, p-value: 7.10e-229, Cramér's V: 0.332


In [6]:
corr, p_value = stats.spearmanr(df["Days_Since_Update"], df["Installs"])
print(f"Spearman correlation (Days_Since_Update vs Installs): {corr:.3f}, p-value: {p_value:.2e}")

Spearman correlation (Days_Since_Update vs Installs): -0.332, p-value: 1.86e-247


In [7]:
corr, p_value = stats.spearmanr(df["Size_MB"].dropna(), 
                                  df.loc[df["Size_MB"].notna(), "Installs"])
print(f"Spearman correlation (Size_MB vs Installs): {corr:.3f}, p-value: {p_value:.2e}")

Spearman correlation (Size_MB vs Installs): 0.310, p-value: 2.13e-187


In [9]:
p_values = {
    "H1 Category (chi2)": 3.53e-161,
    "H2 Free vs Paid (Mann-Whitney)": 3.34e-116,
    "H3 Update recency (chi2)": 7.10e-229,
    "H3 Update recency (Spearman)": 1.86e-247,
    "H4 Size (Spearman)": 2.13e-187,
}

alpha = 0.05
n_tests = len(p_values)
bonferroni_alpha = alpha / n_tests

print(f"Bonferroni-corrected significance threshold: {bonferroni_alpha:.5f}")
print()
for name, p in p_values.items():
    sig = "significant" if p < bonferroni_alpha else "not significant"
    print(f"{name}: p = {p:.2e} → {sig}")

Bonferroni-corrected significance threshold: 0.01000

H1 Category (chi2): p = 3.53e-161 → significant
H2 Free vs Paid (Mann-Whitney): p = 3.34e-116 → significant
H3 Update recency (chi2): p = 7.10e-229 → significant
H3 Update recency (Spearman): p = 1.86e-247 → significant
H4 Size (Spearman): p = 2.13e-187 → significant


## Statistical Test Results

| Hypothesis | Test | Statistic | p-value | Effect size | Verdict |
|---|---|---|---|---|---|
| H1: Category affects success | Chi-square | χ²(32) = 865.28 | 3.53e-161 | Cramér's V = 0.299 | Confirmed |
| H2: Free > Paid installs | Mann-Whitney U | U = 5,031,686 | 3.34e-116 | rank-biserial = 0.499 | Confirmed |
| H3: Recency affects success | Chi-square | χ² = 1063.22 | 7.10e-229 | Cramér's V = 0.332 | Confirmed |
| H3: Recency vs Installs | Spearman | ρ = -0.332 | 1.86e-247 | moderate negative | Confirmed |
| H4: Size vs Installs | Spearman | ρ = 0.310 | 2.13e-187 | moderate positive | Confirmed (reverses original hypothesis) |

**H5 (winner-takes-all)** was answered descriptively in Step 4 (SQL) as a market-concentration measure, not a two-group test.

**Multiple testing correction:** Bonferroni-adjusted α = 0.01. All five results remain significant after correction, none are false positives from running multiple tests.

**Interpretation:** with n = 9,658, nearly every test reaches significance; the effect sizes (Cramér's V 0.30-0.33, rank-biserial 0.50) are what confirm these are practically meaningful differences, not just statistically detectable ones.